# Model Training Pipeline for advanced models

In [1]:
from pathlib import Path
import plotly.graph_objects as go
import pandas as pd
import json
import sys

sys.path.insert(0, "../../")
from geometric_extraction_helper import ALL_FEATURE_KEYS, GENERAL_KEYS, AABB_KEYS, TFBB_KEYS, TOPO_KEYS, MATERIAL_KEYS, RAY_KEYS
from dataviz_helper import plot_feature_importance
from models_helper import ModelTrainer, FeatureGroupEvaluator, collect_model_metrics

In [2]:
DATA_DIR = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"

df_train = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed.parquet")
df_train_over = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed_oversampled.parquet")
df_val   = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

df_train.head()

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,minor_label_unter_terrain,minor_label_konstruktionsergaenzung,minor_label_deckbelag,minor_label_bekleidung,minor_label_aussenliegendes_bauteil,minor_label_sonnenschutz
0,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,-0.752578,0.00,0.0,9.893693,14.360000,2.82,10.646271,14.360000,2.82,0.264881,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,0.000000,-1.85,0.0,10.646270,12.510000,2.82,10.646270,14.360000,2.82,0.264882,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,-0.158796,0.00,0.0,9.394583,9.845000,0.24,9.553379,9.845000,0.24,0.025122,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown


In [3]:
FEATURE_GROUPS = {
    "geom": GENERAL_KEYS,
    "aabb": AABB_KEYS,
    "tfbb": TFBB_KEYS,
    "topo": TOPO_KEYS,
    "material": MATERIAL_KEYS,
    "ray": RAY_KEYS
}

ADVANCED_MODELS = [
    ("gradient_boosting", "Gradient Boosting"),
    ("hist_gradient_boosting", "Hist Gradient Boosting"),
    ("lightgbm", "LightGBM"),
    ("xgboost", "XGBoost"),
]

In [4]:
label_cols = [c for c in df_train.columns if c.startswith("label_")]

X_train = df_train[ALL_FEATURE_KEYS]
y_train = df_train[label_cols]
X_train_over = df_train_over[ALL_FEATURE_KEYS]
y_train_over = df_train_over[label_cols]
X_val   = df_val[ALL_FEATURE_KEYS]
y_val   = df_val[label_cols]

print(f"Train: {len(X_train)}, Train (oversampled): {len(X_train_over)}, Val: {len(X_val)}")
print(f"Labels: {label_cols}")

Train: 26977, Train (oversampled): 28756, Val: 8458
Labels: ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


## 1. Train Models

### 1.1 GradientBoosting

In [4]:
# slower gradient boosting from sklearn but well-established
trainer_gb = ModelTrainer("gradient_boosting", label_cols)
trainer_gb.fit(X_train, y_train)
trainer_gb.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9985,0.9977,0.7067,0.5413,0.9981,0.9974,0.6998,0.5618
label_predefined_type,0.9654,0.9670,0.4815,0.4859,0.9687,0.9829,0.5090,0.5031
label_is_external,0.9501,0.9756,0.7742,0.8847,0.9522,0.9767,0.7817,0.8885
label_load_bearing,0.9406,0.9461,0.7274,0.8072,0.9406,0.9494,0.7040,0.7835
mean,0.9636,0.9716,0.6724,0.6798,0.9649,0.9766,0.6736,0.6842


### 1.2 HistGradientBoosting

In [5]:
# alternative to GradientBoostingClassifier from sklearn -> inspired by LightGBM
trainer_hgb = ModelTrainer("hist_gradient_boosting", label_cols)
trainer_hgb.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.8859,0.7192,0.5402,0.3587,0.9940,0.9884,0.6979,0.5353
label_predefined_type,0.9496,0.8953,0.4530,0.4413,0.9597,0.9454,0.4964,0.4906
label_is_external,0.9980,0.9990,0.7655,0.8811,0.9974,0.9988,0.7509,0.8722
label_load_bearing,0.9978,0.9981,0.7249,0.8170,0.9972,0.9975,0.7336,0.8229
mean,0.9578,0.9029,0.6209,0.6245,0.9871,0.9825,0.6697,0.6802


### 1.3 LightGBM

In [6]:
# original LightGBM library
trainer_lgbm = ModelTrainer("lightgbm", label_cols)
trainer_lgbm.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.5425,0.3467,0.3626,0.2691,1.0000,0.9999,0.7447,0.6037
label_predefined_type,0.3118,0.2205,0.0899,0.1177,0.1517,0.0880,0.0629,0.0569
label_is_external,0.9959,0.9981,0.7554,0.8729,0.9959,0.9980,0.7642,0.8795
label_load_bearing,0.9957,0.9961,0.7248,0.8159,0.9960,0.9965,0.7143,0.8077
mean,0.7115,0.6404,0.4832,0.5189,0.7859,0.7706,0.5715,0.5870


### 1.4 XGBoost

In [7]:
# original XGBoost library
trainer_xgb = ModelTrainer("xgboost", label_cols)
trainer_xgb.compare_training_sets(X_train, y_train, X_train_over, y_train_over, X_val, y_val)

,train_reg_mcc,train_reg_f1_macro,val_reg_mcc,val_reg_f1_macro,train_over_mcc,train_over_f1_macro,val_over_mcc,val_over_f1_macro
label,,,,,,,,
label_ifc_entity,0.9999,0.9999,0.7309,0.5664,1.0000,0.9999,0.7544,0.6112
label_predefined_type,0.9993,0.9998,0.5067,0.5084,0.9992,0.9997,0.5378,0.5661
label_is_external,0.9993,0.9996,0.7283,0.8593,0.9995,0.9997,0.7324,0.8616
label_load_bearing,0.9992,0.9993,0.7333,0.8218,0.9985,0.9987,0.7414,0.8285
mean,0.9994,0.9996,0.6748,0.6890,0.9993,0.9995,0.6915,0.7168


## 2. Train XGBoost and extract Feature Importance

In [5]:
trainer_xgb = ModelTrainer("xgboost", label_cols)
trainer_xgb.fit(X_train_over, y_train_over)
fi_xgb = trainer_xgb.feature_importances(ALL_FEATURE_KEYS)

In [6]:
# plot feature importance for xgboost
df_imp = trainer_xgb.feature_importances(ALL_FEATURE_KEYS)
plot_feature_importance(df_imp, FEATURE_GROUPS, title="XGBoost Feature Importance")

In [7]:
# save feature importance to json
fi_xgb.to_json("feature_importance_xgboost.json", orient="records", indent=2)

## 3. Try different Feature Group Combinations with best model (XGBoost)

In [8]:
# feature group evaluation
evaluator = FeatureGroupEvaluator(FEATURE_GROUPS, label_cols)
results = evaluator.evaluate(X_train_over, y_train_over, X_val, y_val, model_fn="xgboost")

# save feature group evaluation results to json (index keeps the combination names)
results.reset_index().to_json("feature_group_evaluation_xgboost.json", orient="records", indent=2)

In [9]:
results.head(n=30)

,n_features,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,mean_f1_macro
features,,,,,,
geom+aabb+tfbb+topo+material+ray,73,0.618,0.565,0.872,0.820,0.719
geom+aabb+topo+material+ray,60,0.611,0.543,0.864,0.814,0.708
aabb+tfbb+topo+material+ray,61,0.630,0.540,0.857,0.800,0.707
geom+aabb+tfbb+material+ray,64,0.600,0.524,0.877,0.819,0.705
aabb+topo+material+ray,48,0.623,0.527,0.861,0.808,0.705
geom+tfbb+material+ray,50,0.584,0.523,0.877,0.837,0.705
geom+tfbb+topo+material+ray,59,0.545,0.549,0.883,0.840,0.704
geom+tfbb+material,48,0.594,0.492,0.890,0.837,0.703
geom+aabb+topo+material,58,0.597,0.511,0.877,0.822,0.702


## 4. Collect metrics of all advanced models for the report

In [13]:
# collect metrics for all advanced models
advanced_metrics = []
for key, display in ADVANCED_MODELS:
    print(f"collecting metrics for {key}...")
    advanced_metrics.append(collect_model_metrics(key, label_cols, X_train_over, y_train_over, X_val, y_val, display_name=display))

with open("advanced_model_metrics.json", "w") as f:
    json.dump(advanced_metrics, f, indent=2)

pd.DataFrame(advanced_metrics).set_index("model")

collecting metrics for gradient_boosting...
collecting metrics for hist_gradient_boosting...
collecting metrics for lightgbm...
collecting metrics for xgboost...


,train_mcc,train_f1_macro,val_mcc,val_f1_macro,val_precision,val_accuracy
model,,,,,,
Gradient Boosting,0.9649,0.9766,0.6736,0.6842,0.8017,0.7516
Hist Gradient Boosting,0.9871,0.9825,0.6697,0.6802,0.8102,0.7483
LightGBM,0.7859,0.7706,0.5715,0.5870,0.6979,0.6480
XGBoost,0.9993,0.9995,0.6915,0.7168,0.8154,0.7645
